# Ajuste de Umbral — Detector de Plagio

Notebook para encontrar el umbral óptimo del filtro matemático usando el **dev set sintético (60%)**.

**Objetivo:** maximizar el **recall** (no perder plagios reales), aceptando cierta caída en precisión.  
El umbral óptimo encontrado aquí se copia luego a `evaluar_sintetico.py` y `evaluar_clough.py`.

### Flujo
1. Carga el JSON de variantes y construye los pares del dev set
2. Ejecuta **solo el filtro matemático** (sin LLM) para todas las similitudes
3. Barre umbrales de 0.1 a 0.9 y calcula Precision, Recall y F-beta (β=2)
4. Muestra la curva y el umbral óptimo


In [ ]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from collections import defaultdict

# Asegúrate de que detector_plagio.py está en el mismo directorio
import sys
sys.path.insert(0, ".")
from detector_plagio import limpiar_texto, calcular_trigramas, calcular_levenshtein_palabras

# ── Parámetros ──────────────────────────────────────────────────────────────
ARCHIVOS_VARIANTES = [
    "variantes_plagio.json",          # ciencias
    "variantes_literatura_template.json",  # literatura (renombra al rellenado)
]
SEED       = 42
RATIO_DEV  = 0.60   # 60% dev, 40% test
MIN_PALABRAS = 15
BETA       = 2      # F-beta con beta=2 prioriza recall sobre precision


In [ ]:
# ── Carga de variantes ──────────────────────────────────────────────────────
NIVELES_PLAGIO    = {"nivel_0", "nivel_1", "nivel_2", "nivel_3"}
NIVELES_NO_PLAGIO = {"nivel_4", "nivel_5"}

def cargar_variantes(rutas):
    combinado = {}
    for ruta in rutas:
        with open(ruta, "r", encoding="utf-8") as f:
            combinado.update(json.load(f))
    return combinado

def construir_pares(variantes):
    pares = []
    for task_id, perfiles in variantes.items():
        for perfil, datos in perfiles.items():
            original = datos.get("respuesta_original", "")
            vars_dict = datos.get("variantes") or {}
            for nivel, texto in vars_dict.items():
                if not texto:
                    continue
                pares.append({
                    "task_id": task_id, "perfil": perfil, "nivel": nivel,
                    "original": original, "variante": texto,
                    "etiqueta": "plagio" if nivel in NIVELES_PLAGIO else "no_plagio",
                })
    return pares

def split_dev(pares, seed=SEED, ratio_dev=RATIO_DEV):
    unidades = list({(p["task_id"], p["perfil"]) for p in pares})
    random.seed(seed)
    random.shuffle(unidades)
    n_dev = int(len(unidades) * ratio_dev)
    unidades_dev = set(map(tuple, [list(u) for u in unidades[:n_dev]]))
    return [p for p in pares if (p["task_id"], p["perfil"]) in unidades_dev]

variantes = cargar_variantes(ARCHIVOS_VARIANTES)
pares     = construir_pares(variantes)
pares_dev = split_dev(pares)

print(f"Total pares construidos : {len(pares)}")
print(f"Pares en dev set (60%)  : {len(pares_dev)}")
print(f"  → Plagios reales      : {sum(1 for p in pares_dev if p['etiqueta'] == 'plagio')}")
print(f"  → No plagios reales   : {sum(1 for p in pares_dev if p['etiqueta'] == 'no_plagio')}")


In [ ]:
# ── Calcular similitudes (sin LLM) ──────────────────────────────────────────
registros = []
for par in pares_dev:
    w_orig = limpiar_texto(par["original"])
    w_var  = limpiar_texto(par["variante"])
    n_orig, n_var = len(w_orig), len(w_var)
    es_corto = n_orig < MIN_PALABRAS or n_var < MIN_PALABRAS

    if es_corto:
        sim = None   # pares cortos no pasan por filtro matemático
    else:
        sim_tri = calcular_trigramas(w_orig, w_var)
        sim_lev = calcular_levenshtein_palabras(w_orig, w_var)
        sim = max(sim_tri, sim_lev)

    registros.append({
        **par,
        "sim_maxima": sim,
        "es_corto": es_corto,
    })

n_cortos = sum(1 for r in registros if r["es_corto"])
print(f"Pares cortos (van directo al LLM, excluidos del barrido): {n_cortos}")
print(f"Pares con similitud calculada: {len(registros) - n_cortos}")


In [ ]:
# ── Barrido de umbrales ─────────────────────────────────────────────────────
# Solo sobre los pares con similitud calculada (excluimos los cortos)
registros_validos = [r for r in registros if not r["es_corto"]]

umbrales   = np.arange(0.10, 0.95, 0.05)
precisiones, recalls, fbetas = [], [], []

for t in umbrales:
    tp = fp = fn = tn = 0
    for r in registros_validos:
        pred    = "plagio" if r["sim_maxima"] >= t else "no_plagio"
        real    = r["etiqueta"]
        if pred == "plagio"    and real == "plagio":    tp += 1
        elif pred == "plagio"  and real == "no_plagio": fp += 1
        elif pred == "no_plagio" and real == "plagio":  fn += 1
        else:                                            tn += 1

    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec    = tp / (tp + fn) if (tp + fn) > 0 else 0
    fb     = (1 + BETA**2) * prec * rec / (BETA**2 * prec + rec) if (prec + rec) > 0 else 0

    precisiones.append(prec)
    recalls.append(rec)
    fbetas.append(fb)

# Umbral óptimo: el que maximiza F-beta
idx_opt   = int(np.argmax(fbetas))
t_optimo  = umbrales[idx_opt]
print(f"Umbral óptimo (max F{BETA}): {t_optimo:.2f}")
print(f"  Precision : {precisiones[idx_opt]:.3f}")
print(f"  Recall    : {recalls[idx_opt]:.3f}")
print(f"  F{BETA}       : {fbetas[idx_opt]:.3f}")


In [ ]:
# ── Gráfica de curvas ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(umbrales, precisiones, "o-", label="Precisión",  color="#2196F3")
ax.plot(umbrales, recalls,     "s-", label="Recall",     color="#4CAF50")
ax.plot(umbrales, fbetas,      "^-", label=f"F{BETA} (β={BETA})", color="#FF9800", linewidth=2)

ax.axvline(t_optimo, color="red", linestyle="--", linewidth=1.5,
           label=f"Umbral óptimo = {t_optimo:.2f}")

ax.set_xlabel("Umbral de similitud", fontsize=12)
ax.set_ylabel("Métrica", fontsize=12)
ax.set_title(f"Curvas Precision / Recall / F{BETA} por umbral (dev set sintético)", fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("curva_umbral.png", dpi=150)
plt.show()
print("Gráfica guardada en curva_umbral.png")


In [ ]:
# ── Desglose por nivel de obfuscación con el umbral óptimo ─────────────────
print(f"Desglose con umbral óptimo = {t_optimo:.2f}\n")
print(f"{'Nivel':<12} {'Total':>6} {'Detectados':>11} {'Tasa detección':>15}")
print("-" * 48)

por_nivel = defaultdict(list)
for r in registros_validos:
    por_nivel[r["nivel"]].append(r)

for nivel in sorted(por_nivel.keys()):
    registros_nivel = por_nivel[nivel]
    detectados = sum(1 for r in registros_nivel if r["sim_maxima"] >= t_optimo)
    total      = len(registros_nivel)
    tasa       = detectados / total if total > 0 else 0
    es_plagio  = nivel in NIVELES_PLAGIO
    marca      = "✓ plagio" if es_plagio else "✗ no plagio"
    print(f"{nivel:<12} {total:>6} {detectados:>11} {tasa:>14.1%}  [{marca}]")

print(f"\n→ Copia este valor en evaluar_sintetico.py y evaluar_clough.py:")
print(f"   UMBRAL = {t_optimo:.2f}")
